### Langkah 2: Preprocessing Data (Konversi Rupiah, Encoding, & Scaling)

Notebook ini berfokus pada persiapan data sebelum masuk ke pelatihan Machine Learning:
1. Load Data Bersih: Membaca dataset german_credit_clean_idr.csv yang sudah disiapkan di langkah 1.
2. Pemisahan Fitur & Target:
   - $X$ = Semua fitur profil nasabah (termasuk pinjaman dalam Rupiah).
   - $y$ = Target kelayakan (risk: 1 = Layak, 0 = Tidak Layak).
3. Encoding Fitur Kategori (One-Hot Encoding):
   - Mengubah teks/kategori (seperti status rumah, tujuan pinjaman, status rekening) menjadi angka biner (0 dan 1) agar bisa dihitung oleh algoritma.
4. Penyelarasan Skala (Feature Scaling via StandardScaler):
   - Menyamakan skala antara angka jutaan rupiah (credit_amount_idr), umur (misal 25 tahun), dan durasi (12 bulan) agar model adil dalam menghitung bobot.
5. Menyimpan Objek Preprocessor (preprocessor.pkl):
   - Sangat penting! Objek transformer ini disimpan agar nanti di aplikasi web, data baru dari nasabah bisa langsung diproses dengan standar yang persis sama.

### 1. Import Library & Load Data

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Deteksi otomatis lokasi file dataset bersih
possible_paths = [
    "german_credit_clean_idr.csv",
    os.path.join("data", "german_credit_clean_idr.csv"),
    os.path.join("..", "data", "german_credit_clean_idr.csv")
]

CSV_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        CSV_PATH = p
        break

if CSV_PATH is None:
    raise FileNotFoundError(f"File german_credit_clean_idr.csv tidak ditemukan. Jalankan notebook 1 terlebih dahulu!")

df = pd.read_csv(CSV_PATH)
print(f"✓ Data berhasil dimuat: {CSV_PATH}")
print(f"✓ Ukuran dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df[['credit_amount_idr', 'duration_month', 'age', 'risk']].head()

✓ Data berhasil dimuat: data\german_credit_clean_idr.csv
✓ Ukuran dataset: 1000 baris, 22 kolom


,credit_amount_idr,duration_month,age,risk
0,11690000,6,67,1
1,59510000,48,22,0
2,20960000,12,49,1
3,78820000,42,45,1
4,48700000,24,53,0


### 2. Memisahkan Fitur Numerik dan Kategorikal

- Kolom credit_amount asli dalam Deutsche Mark (DM) kita drop karena sudah digantikan oleh credit_amount_idr.
- Kolom target risk dipisahkan sebagai label $y$.

In [2]:
# Pisahkan Target (y) dan Fitur (X)
y = df['risk']
X = df.drop(columns=['risk', 'credit_amount'])  # Kita gunakan credit_amount_idr

# Identifikasi kolom numerik dan kategorikal
numeric_features = [
    'duration_month',
    'credit_amount_idr',   # Pinjaman dalam format Rupiah
    'installment_rate',
    'residence_since',
    'age',
    'existing_credits',
    'num_dependents'
]

categorical_features = [col for col in X.columns if col not in numeric_features]

print(f"Jumlah Fitur Numerik    : {len(numeric_features)}")
print(f"Jumlah Fitur Kategorikal: {len(categorical_features)}")
print("\nFitur Numerik:", numeric_features)
print("\nFitur Kategorikal:", categorical_features)

Jumlah Fitur Numerik    : 7
Jumlah Fitur Kategorikal: 13

Fitur Numerik: ['duration_month', 'credit_amount_idr', 'installment_rate', 'residence_since', 'age', 'existing_credits', 'num_dependents']

Fitur Kategorikal: ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment_length', 'personal_status_sex', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']


### 3. Membangun ColumnTransformer (Scaling + One-Hot Encoding)

Menggunakan ColumnTransformer dari Scikit-Learn:
1. StandardScaler() untuk fitur numerik $\rightarrow$ mengubah rata-rata menjadi 0 dan standar deviasi 1.
2. OneHotEncoder(drop='first') untuk fitur kategorikal $\rightarrow$ mengubah teks menjadi kolom biner dummy (0 atau 1) tanpa jebakan multikolinearitas.

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

# Terapkan preprocessing pada seluruh data X
X_preprocessed = preprocessor.fit_transform(X)

# Dapatkan nama fitur setelah di-encode
encoded_cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(encoded_cat_names)

print(f"Bentuk data sebelum preprocessing: {X.shape}")
print(f"Bentuk data setelah preprocessing : {X_preprocessed.shape}")
print(f"Total fitur yang dihasilkan       : {len(all_feature_names)} fitur angka")

Bentuk data sebelum preprocessing: (1000, 20)
Bentuk data setelah preprocessing : (1000, 48)
Total fitur yang dihasilkan       : 48 fitur angka


### 4. Menampilkan Contoh Hasil Data yang Sudah Siap untuk Model

In [4]:
# Buat DataFrame dari hasil olahan agar mudah dilihat
df_preprocessed = pd.DataFrame(X_preprocessed, columns=all_feature_names)
print("Cuplikan 5 baris pertama data hasil preprocessing (semua sudah angka murni & berskala adil):")
df_preprocessed.head()

Cuplikan 5 baris pertama data hasil preprocessing (semua sudah angka murni & berskala adil):


,duration_month,credit_amount_idr,installment_rate,residence_since,age,existing_credits,num_dependents,checking_status_A12,checking_status_A13,checking_status_A14,...,property_A124,other_installment_plans_A142,other_installment_plans_A143,housing_A152,housing_A153,job_A172,job_A173,job_A174,telephone_A192,foreign_worker_A202
0,-1.236478,-0.745131,0.918477,1.046987,2.766456,1.027079,-0.428290,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1,2.248194,0.949817,-0.870183,-0.765977,-1.191404,-0.704926,-0.428290,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2,-0.738668,-0.416562,-0.870183,0.140505,1.183312,-0.704926,2.334869,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1.750384,1.634247,-0.870183,1.046987,0.831502,-0.704926,2.334869,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,0.256953,0.566664,0.024147,1.046987,1.535122,1.027079,2.334869,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


### 5. Simpan Preprocessor dan Data Siap Latih

1. preprocessor.pkl: Berisi rumus standarisasi dan pemetaan kategori.
2. preprocessed_data.npz: Menyimpan matriks $X$ dan $y$ yang sudah siap langsung dipakai di notebook training.

In [5]:
# Tentukan direktori penyimpanan
OUTPUT_DIR = os.path.dirname(CSV_PATH) if os.path.dirname(CSV_PATH) != "" else "."

# 1. Simpan objek preprocessor
PREPROCESSOR_PATH = os.path.join(OUTPUT_DIR, "preprocessor.pkl")
joblib.dump(preprocessor, PREPROCESSOR_PATH)
print(f"✓ Preprocessor berhasil disimpan di: {PREPROCESSOR_PATH}")

# 2. Simpan X_preprocessed dan y untuk notebook training
DATA_NPZ_PATH = os.path.join(OUTPUT_DIR, "preprocessed_data.npz")
np.savez_compressed(DATA_NPZ_PATH, X=X_preprocessed, y=y.values, feature_names=np.array(all_feature_names))
print(f"✓ Data siap latih berhasil disimpan di: {DATA_NPZ_PATH}")
print("\nSemua tahap preprocessing selesai! Siap masuk ke tahap modeling.")

✓ Preprocessor berhasil disimpan di: data\preprocessor.pkl
✓ Data siap latih berhasil disimpan di: data\preprocessed_data.npz

Semua tahap preprocessing selesai! Siap masuk ke tahap modeling.
